# Olist Price Optimizer — Step 2: Price Elasticity Modelling

**Goal:** Estimate how sensitive demand (sales volume) is to price changes —
both globally (baseline) and per product category — then identify the price
point that maximises monthly revenue for each category.

**Input:** `data/category_month_agg.parquet` — 1,241 rows, 70 categories, 23 months  
**Outputs:**
* `data/elasticity_by_category.parquet / .csv` — per-category elasticity estimates
* Revenue curve analysis for the top 5 highest-volume categories

---
### Analytical Framework

We use a **log-log Ordinary Least Squares (OLS)** regression:

```
log(sales) = α + β·log(price) + γ·log(freight) + δ·avg_review
           + ζ·late_delivery_rate + Σ month_dummies + ε
```

The coefficient **β** is the **price elasticity of demand**: a 1% increase in
price leads to a β% change in monthly sales volume. Key thresholds:

| β range | Demand type | Revenue implication |
|---------|-------------|---------------------|
| β < −1 | Elastic | Raising price **reduces** revenue |
| β = −1 | Unit-elastic | Price changes leave revenue unchanged |
| −1 < β < 0 | Inelastic | Raising price **increases** revenue |
| β > 0 | Veblen/quality signal | Unusual; may indicate omitted variables |

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

DATA_DIR = Path("../data")
print("Libraries loaded.")

Libraries loaded.


---
## Section 1 — Setup & Load

### Why filter to 2017-01?

The Olist marketplace launched in late 2016. Those early months have very few
active categories (1–30 per month vs. 60–70 from 2017 onward), and prices and
volumes reflect early-adopter behaviour rather than a stable market.
Including them would add noise and bias the elasticity estimate.
We keep **2017-01 onward** — the point where most categories reach consistent
monthly activity (≥ 40 categories reporting every month).

In [2]:
df = pd.read_parquet(DATA_DIR / "category_month_agg.parquet")
df["year_month"] = pd.PeriodIndex(df["year_month"], freq="M")

print(f"Full dataset:  {df.shape[0]:,} rows  |  {df['product_category_name'].nunique()} categories")
print(f"Date range:    {df['year_month'].min()} to {df['year_month'].max()}")
print()

df_model = df[df["year_month"] >= pd.Period("2017-01", "M")].copy()
print(f"After filter:  {df_model.shape[0]:,} rows  |  {df_model['product_category_name'].nunique()} categories")
print(f"Date range:    {df_model['year_month'].min()} to {df_model['year_month'].max()}")
print()
print("Monthly sales distribution (items per category per month):")
print(df_model["sales"].describe().round(1).to_string())

Full dataset:  1,241 rows  |  70 categories
Date range:    2016-09 to 2018-08

After filter:  1,210 rows  |  70 categories
Date range:    2017-01 to 2018-08

Monthly sales distribution (items per category per month):
count    1210.0
mean       88.8
std       152.2
min         1.0
25%         6.0
50%        20.0
75%       101.0
max       970.0


---
## Section 2 — Log Transformation

### Why log-log?

Price-demand relationships are multiplicative, not additive. A R\$10 increase
means something very different for a R\$20 product vs. a R\$500 product.
Taking logarithms converts the relationship to **percentage changes**, making
the OLS coefficient directly interpretable as elasticity:

> **"A 1% increase in average price leads to a β% change in monthly sales volume."**

We use `np.log1p(x) = log(1 + x)` instead of `log(x)` to guard against any
zero values that may appear in future data refreshes.

| Original | Log version | Role in model |
|----------|-------------|---------------|
| `sales` | `log_sales` | Dependent variable (demand proxy) |
| `avg_price` | `log_price` | Key predictor — gives us β (elasticity) |
| `avg_freight` | `log_freight` | Confounder: high freight inflates perceived total cost |
| `avg_distance` | `log_distance` | Diagnostic; correlated with freight |

In [3]:
for col, new_col in [
    ("sales",        "log_sales"),
    ("avg_price",    "log_price"),
    ("avg_freight",  "log_freight"),
    ("avg_distance", "log_distance"),
]:
    df_model[new_col] = np.log1p(df_model[col])

log_cols = ["log_sales", "log_price", "log_freight", "log_distance"]
print("Descriptive stats for log-transformed columns:")
print(df_model[log_cols].describe().round(3).to_string())
print()
show = ["product_category_name", "year_month", "avg_price", "log_price", "sales", "log_sales"]
print("Sample: original vs. log-transformed values:")
print(df_model[show].head(5).to_string(index=False))

Descriptive stats for log-transformed columns:
       log_sales  log_price  log_freight  log_distance
count   1210.000   1210.000     1210.000      1210.000
mean       3.221      4.354        2.945         6.013
std        1.667      0.767        0.345         0.685
min        0.693      1.589        0.104         1.785
25%        1.946      3.930        2.780         5.874
50%        3.045      4.369        2.864         6.072
75%        4.625      4.754        2.989         6.275
max        6.878      8.175        4.505         7.948

Sample: original vs. log-transformed values:
     product_category_name year_month  avg_price  log_price  sales  log_sales
agro_industry_and_commerce    2017-01     21.990   3.135059      3   1.386294
agro_industry_and_commerce    2017-02     21.990   3.135059      7   2.079442
agro_industry_and_commerce    2017-03     40.995   3.737551      2   1.098612
agro_industry_and_commerce    2017-05    324.990   5.786867      4   1.609438
agro_industry_and_comm

---
## Section 3 — Global Elasticity Model (Baseline)

We start with a **pooled OLS** across all categories and months.
This gives us the average price elasticity of the entire Olist platform —
a useful benchmark before we disaggregate by category.

**Model specification:**
```
log_sales ~ log_price + log_freight + avg_review + late_delivery_rate + C(month)
```

* `C(month)` adds 11 binary dummies (reference = January) to absorb calendar
  seasonality. Without this, β would conflate price changes with seasonal swings.
* `log_freight` controls for the fact that high-freight items often have higher
  prices — omitting it would bias β toward zero or inflate it.
* `avg_review` and `late_delivery_rate` capture service quality, which affects
  repeat purchases and may correlate with price positioning.

**Why C(month) works here but not per-category:**  
The pooled dataset has 1,210 rows — far more than the 16 parameters in the model.
For per-category models (Section 4), each category has at most 20 monthly
observations, exactly equal to the number of months; adding month dummies would
exhaust all degrees of freedom. Section 4 therefore uses the model *without*
`C(month)`, isolating the pure price signal within each category.

In [4]:
GLOBAL_FORMULA = (
    "log_sales ~ log_price + log_freight + avg_review "
    "+ late_delivery_rate + C(month)"
)
CAT_FORMULA = (
    "log_sales ~ log_price + log_freight + avg_review + late_delivery_rate"
)

global_model = smf.ols(GLOBAL_FORMULA, data=df_model).fit()
print(global_model.summary())

                            OLS Regression Results                            
Dep. Variable:              log_sales   R-squared:                       0.017
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     1.399
Date:                Sat, 09 May 2026   Prob (F-statistic):              0.139
Time:                        18:08:37   Log-Likelihood:                -2324.5
No. Observations:                1210   AIC:                             4681.
Df Residuals:                    1194   BIC:                             4763.
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              4.8992      0

In [5]:
elas  = global_model.params["log_price"]
pval  = global_model.pvalues["log_price"]
ci    = global_model.conf_int()
ci_lo = ci.loc["log_price"].iloc[0]
ci_hi = ci.loc["log_price"].iloc[1]

if pval < 0.01:
    sig_label = "*** (p < 0.01)"
elif pval < 0.05:
    sig_label = "** (p < 0.05)"
elif pval < 0.10:
    sig_label = "* (p < 0.10)"
else:
    sig_label = "(not significant)"

print("=" * 56)
print("  GLOBAL PRICE ELASTICITY")
print("=" * 56)
print(f"  Elasticity (beta):  {elas:+.4f}  {sig_label}")
print(f"  95% CI:             [{ci_lo:.4f},  {ci_hi:.4f}]")
print(f"  R-squared:          {global_model.rsquared:.4f}")
print(f"  N observations:     {int(global_model.nobs):,}")
print("=" * 56)
print()
print(f"  Interpretation: a 1% increase in avg_price is associated")
print(f"  with a {elas:+.2f}% change in monthly sales volume,")
print(f"  holding freight, review score, and seasonality constant.")

  GLOBAL PRICE ELASTICITY
  Elasticity (beta):  +0.0483  (not significant)
  95% CI:             [-0.0962,  0.1927]
  R-squared:          0.0173
  N observations:     1,210

  Interpretation: a 1% increase in avg_price is associated
  with a +0.05% change in monthly sales volume,
  holding freight, review score, and seasonality constant.


### Business Interpretation of the Global Model

The global coefficient is an average across all product categories and
all Olist sellers. It tells us the baseline price sensitivity of the platform.

| Scenario | What it means for pricing strategy |
|----------|-------------------------------------|
| **β < −1** (elastic) | Platform demand is price-sensitive. Promotions drive disproportionate volume; raising prices may shrink total GMV. |
| **−1 < β < 0** (inelastic) | Demand is relatively stable to price. Modest increases grow revenue with limited volume loss. |
| **β ≈ 0** | Price is not the main demand driver at a platform level. Quality, assortment, and logistics dominate. |
| **β > 0** | May indicate positive brand/quality signalling, or an omitted variable (e.g., product newness, seasonality). |

> **Caution:** This is an observational estimate, not a controlled experiment.
> Sellers who raise prices may simultaneously improve their assortment or
> invest in marketing, which would bias β upward. Treat as a directional signal.

---
## Section 4 — Elasticity by Category

The global estimate masks substantial heterogeneity: a luxury-electronics buyer
behaves very differently from someone buying bed linen or garden tools.
We now fit the category-level OLS for each category with ≥ 12 monthly obs:

```
log_sales ~ log_price + log_freight + avg_review + late_delivery_rate
```

Month dummies are excluded here because each category has at most 20 monthly
rows — adding 19 dummies would leave zero degrees of freedom. The seasonality
is already captured in the global model; here we isolate the price signal.

Results are stored with:
* `price_elasticity` — the `log_price` coefficient β for that category
* `p_value` — statistical significance (H₀: β = 0)
* `r_squared` — how well the 4-variable model explains demand variation
* `significant` — True if p_value < 0.05

In [6]:
records = []
skipped = 0

for cat, grp in df_model.groupby("product_category_name"):
    if len(grp) < 12:
        skipped += 1
        continue
    try:
        m = smf.ols(CAT_FORMULA, data=grp).fit(disp=False)
        elas_val = m.params.get("log_price", np.nan)
        pval_val = m.pvalues.get("log_price", np.nan)
        if pd.isna(elas_val):
            skipped += 1
            continue
        records.append({
            "category":         cat,
            "price_elasticity": elas_val,
            "p_value":          pval_val,
            "r_squared":        m.rsquared,
            "n_obs":            int(m.nobs),
            "significant":      pval_val < 0.05,
        })
    except Exception:
        skipped += 1

elasticity_df = (
    pd.DataFrame(records)
    .sort_values("price_elasticity")
    .reset_index(drop=True)
)

print(f"Categories modelled:            {len(elasticity_df)}")
print(f"Skipped (< 12 obs or error):    {skipped}")
print(f"Significant (p < 0.05):         {elasticity_df['significant'].sum()}")
print(f"Negative elasticity (expected): {(elasticity_df['price_elasticity'] < 0).sum()}")
print(f"Positive elasticity (unusual):  {(elasticity_df['price_elasticity'] > 0).sum()}")
print()
print("Elasticity distribution:")
print(elasticity_df["price_elasticity"].describe().round(3).to_string())

Categories modelled:            62
Skipped (< 12 obs or error):    8
Significant (p < 0.05):         13
Negative elasticity (expected): 40
Positive elasticity (unusual):  22

Elasticity distribution:
count    62.000
mean     -0.242
std       0.946
min      -3.060
25%      -0.688
50%      -0.247
75%       0.311
max       2.450


In [7]:
HDR = (
    f"  {'Category':<42s} {'Elasticity':>11s}  {'p-val':>7s}  "
    f"{'R2':>6s}  {'n':>4s}  {'Sig?':>5s}"
)
SEP = "  " + chr(8212) * 80


def print_rows(subset):
    for _, r in subset.iterrows():
        sig = "yes" if r["significant"] else "no "
        print(
            f"  {r['category']:<42s} {r['price_elasticity']:>11.3f}  "
            f"{r['p_value']:>7.4f}  {r['r_squared']:>6.3f}  "
            f"{int(r['n_obs']):>4}  {sig:>5s}"
        )


print("TOP 10 MOST PRICE-ELASTIC (highest demand sensitivity to price):")
print(HDR); print(SEP)
print_rows(elasticity_df.head(10))

print()
print("TOP 10 LEAST PRICE-ELASTIC (demand least sensitive to price):")
print(HDR); print(SEP)
print_rows(elasticity_df.tail(10).iloc[::-1])

TOP 10 MOST PRICE-ELASTIC (highest demand sensitivity to price):
  Category                                    Elasticity    p-val      R2     n   Sig?
  ————————————————————————————————————————————————————————————————————————————————
  watches_gifts                                   -3.060   0.0000   0.771    20    yes
  industry_commerce_and_business                  -2.317   0.0057   0.658    18    yes
  electronics                                     -2.002   0.0002   0.676    20    yes
  auto                                            -1.941   0.0022   0.629    20    yes
  furniture_living_room                           -1.882   0.0058   0.485    20    yes
  garden_tools                                    -1.295   0.0114   0.589    20    yes
  home_confort                                    -1.269   0.1604   0.368    19    no 
  computers                                       -1.187   0.1023   0.381    13    no 
  office_furniture                                -1.185   0.3904   0

### What Elasticity Means for Commercial Decisions

| Category type | Elasticity range | Recommended action |
|---------------|------------------|-----------|
| **Highly elastic** (β < −1.5) | Price-sensitive buyers | Do not raise price; grow volume through promotions or bundles |
| **Moderately elastic** (−1.5 ≤ β < −1) | Meaningful price sensitivity | Price near the revenue-maximising point (see Section 5) |
| **Inelastic** (−1 ≤ β < 0) | Stable demand | Gradual increases grow revenue with minimal volume loss |
| **Near-zero / positive** | Price not main driver | Invest in quality signals: photos, review score, faster delivery |

> **Statistical caveat:** Only categories with `significant = True` (p < 0.05)
> have reliable elasticity estimates. Treat insignificant results as directional
> signals only — the confidence interval likely spans zero.

---
## Section 5 — Revenue Curve per Category

Given an elasticity estimate, we can simulate monthly revenue as price moves
away from its current average. The log-log model implies a power-law demand:

```
Q(P) = Q_base * (P / P_base)^β
Revenue(P) = P * Q(P)
```

The analytical revenue-maximising price is **P\* = −P_base · β / (1 + β)**
when β < −1 (elastic demand). For inelastic categories (β > −1), revenue
increases monotonically with price — in the simulation the curve will peak at
the 200% boundary, signalling that modest price increases are safe.

We run this for the **top 5 categories by total sales volume** — those where
pricing decisions have the highest absolute revenue impact on the platform.

In [8]:
def build_revenue_curve(
    category: str,
    elasticity: float,
    base_price: float,
    base_sales: float,
    n_points: int = 200,
) -> tuple:
    # Simulate prices from 50% to 200% of the current average price
    prices = np.linspace(base_price * 0.50, base_price * 2.0, n_points)
    # Power-law demand: Q(P) = Q_base * (P / P_base)^elasticity
    estimated_sales = base_sales * (prices / base_price) ** elasticity
    revenue = prices * estimated_sales
    df_curve = pd.DataFrame({
        "price":           prices,
        "estimated_sales": estimated_sales,
        "revenue":         revenue,
    })
    optimal_price = df_curve.loc[df_curve["revenue"].idxmax(), "price"]
    return df_curve, optimal_price

In [9]:
top5_cats = (
    df_model.groupby("product_category_name")["sales"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index.tolist()
)
print(f"Top 5 categories by total sales: {top5_cats}")
print()

HDR5 = (
    f"{'Category':<28s}  {'Elasticity':>10s}  {'Base P':>8s}  {'Sales/mo':>8s}  "
    f"{'Opt Price':>9s}  {'Curr Rev':>9s}  {'Max Rev':>9s}  {'Uplift':>7s}  {'Type':>9s}"
)
print(HDR5)
print(chr(8212) * 115)

curve_results = {}
for cat in top5_cats:
    row = elasticity_df[elasticity_df["category"] == cat]
    if row.empty:
        print(f"{cat:<28s}  [no elasticity estimate]")
        continue

    elasticity   = row["price_elasticity"].values[0]
    cat_data     = df_model[df_model["product_category_name"] == cat]
    base_price   = cat_data["avg_price"].mean()
    base_sales   = cat_data["sales"].mean()

    df_curve, opt_price = build_revenue_curve(cat, elasticity, base_price, base_sales)
    curve_results[cat] = df_curve

    curr_rev   = base_price * base_sales
    max_rev    = df_curve["revenue"].max()
    uplift     = (max_rev - curr_rev) / curr_rev * 100
    elas_type  = "elastic" if elasticity < -1 else "inelastic"

    print(
        f"{cat:<28s}  {elasticity:>10.3f}  {base_price:>8.2f}  {base_sales:>8.1f}  "
        f"{opt_price:>9.2f}  {curr_rev:>9.0f}  {max_rev:>9.0f}  "
        f"{uplift:>6.1f}%  {elas_type:>9s}"
    )

Top 5 categories by total sales: ['bed_bath_table', 'health_beauty', 'sports_leisure', 'furniture_decor', 'computers_accessories']

Category                      Elasticity    Base P  Sales/mo  Opt Price   Curr Rev    Max Rev   Uplift       Type
———————————————————————————————————————————————————————————————————————————————————————————————————————————————————
bed_bath_table                     0.273     79.99     545.0     159.99      43601     105373   141.7%  inelastic
health_beauty                      1.244     76.99     469.6     153.98      36154     171272   373.7%  inelastic
sports_leisure                     2.450     76.62     419.9     153.24      32168     351559   992.9%  inelastic
furniture_decor                    0.167     62.38     398.9     124.76      24887      55867   124.5%  inelastic
computers_accessories              1.736     75.47     380.5     150.94      28717     191311   566.2%  inelastic


### Reading the Revenue Curve Table

* **Curr Rev** = `base_price × avg_sales_per_month` — the current revenue baseline.
* **Max Rev** = peak of the simulated curve — achievable at the **Opt Price**.
* **Uplift** = `(Max Rev − Curr Rev) / Curr Rev` — potential revenue gain from
  repricing, assuming the demand curve is stable.

| Demand type | What the curve looks like | Practical meaning |
|-------------|--------------------------|-------------------|
| **Elastic** (β < −1) | Inverted-U with clear interior peak | Move price toward Opt Price for maximum revenue |
| **Inelastic** (β > −1) | Monotonically increasing | Price can be raised; Opt Price = simulation upper bound |

> These curves are **model-based projections**, not guarantees. Real markets
> are affected by competitor pricing, stock availability, and marketing spend.
> Use as a directional tool, validated against A/B price tests.

---
## Section 6 — Save Results

We persist the per-category elasticity table in two formats:
* **Parquet** — efficient binary format for the Streamlit app and notebook 03.
* **CSV** — human-readable for quick inspection in Excel or a text editor.

The table is sorted by `price_elasticity` (most elastic first) so that
the highest-priority repricing opportunities appear at the top.

In [10]:
parquet_path = DATA_DIR / "elasticity_by_category.parquet"
csv_path     = DATA_DIR / "elasticity_by_category.csv"

elasticity_df.to_parquet(parquet_path, index=False)
elasticity_df.to_csv(csv_path, index=False)

print(f"Saved: {parquet_path.resolve()}")
print(f"Saved: {csv_path.resolve()}")
print(f"Shape: {elasticity_df.shape}")
print()
print("First 10 rows (most elastic → least elastic):")
print(elasticity_df.head(10).to_string(index=False))

Saved: C:\Users\lpraz\OneDrive\Área de Trabalho\Masters Degree\02 - Portfolio\03 - Growth\02 - Olist price elasticity\data\elasticity_by_category.parquet
Saved: C:\Users\lpraz\OneDrive\Área de Trabalho\Masters Degree\02 - Portfolio\03 - Growth\02 - Olist price elasticity\data\elasticity_by_category.csv
Shape: (62, 6)

First 10 rows (most elastic → least elastic):
                      category  price_elasticity  p_value  r_squared  n_obs  significant
                 watches_gifts         -3.060168 0.000006   0.771473     20         True
industry_commerce_and_business         -2.317128 0.005654   0.657998     18         True
                   electronics         -2.002468 0.000202   0.676396     20         True
                          auto         -1.940704 0.002193   0.628906     20         True
         furniture_living_room         -1.881933 0.005849   0.484874     20         True
                  garden_tools         -1.294905 0.011367   0.589228     20         True
           